In [ ]:
"""
Author:     Aaron Niecestro
Project:    Deep Learning for Wine Quality (DLWQ)

Created:    November 2, 2026
Last Edit:  November 2, 2026
Progress:   Complete

Analysis:   06 - Optimizer Benchmarking
Method:     RMSProp

Description:
This script evaluates the winning optimizer identified
during Analysis 6 and establishes the standalone
optimizer benchmark for future analyses.

Winning Configuration:

- StandardScaler
- Sigmoid Activation
- Architecture: 64 -> 32 -> 16
- RMSProp Optimizer
- 125 Epochs

Research Question:
Can RMSProp provide the strongest optimization
performance for wine quality prediction?
"""

==========================================
Imports
==========================================

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import Input

In [ ]:
from tensorflow.keras.optimizers import RMSprop

------------------------------------------
Reproducibility
------------------------------------------

In [ ]:
import os
import random
import tensorflow as tf

In [ ]:
SEED = 42

In [ ]:
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

==========================================
Load Data
==========================================

In [ ]:
file_path = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\deep-learning-rl\00-data\winequality-final.csv"
)

In [ ]:
output_dir = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\deep-learning-rl\07-Optimizer-Benchmarking\tables_figures"
)

In [ ]:
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
dpwq = pd.read_csv(
    file_path
)

In [ ]:
dpwq.columns = (
    dpwq.columns
    .str.strip()
)

==========================================
Features and Target
==========================================

In [ ]:
X = dpwq.drop(
    "quality",
    axis=1
)

In [ ]:
y = dpwq["quality"]

==========================================
Encode Wine Type
==========================================

In [ ]:
X = pd.get_dummies(
    X,
    columns=["type"],
    drop_first=True
)

In [ ]:
X["type_white"] = (
    X["type_white"]
    .astype(int)
)

==========================================
Train Test Split
==========================================

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y
)

==========================================
StandardScaler Scaling
==========================================

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(
    X_train
)

In [ ]:
X_test_scaled = scaler.transform(
    X_test
)

==========================================
Helper Functions
==========================================

In [ ]:
def adjusted_r2(r2, n, p):

    return (
        1 -
        (((1 - r2) * (n - 1))
         / (n - p - 1))
    )

In [ ]:
def evaluate_model(
    y_true,
    y_pred,
    p
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    adj_r2 = adjusted_r2(
        r2,
        len(y_true),
        p
    )

    return (
        mae,
        rmse,
        adj_r2
    )

==========================================
RMSProp Model
==========================================

In [ ]:
model = Sequential([
    
    tf.keras.backend.clear_session(),

    Input(
        shape=(
            X_train_scaled.shape[1],
        )
    ),

    Dense(
        64,
        activation="sigmoid"
    ),

    Dense(
        32,
        activation="sigmoid"
    ),

    Dense(
        16,
        activation="sigmoid"
    ),

    Dense(1)

In [ ]:
])

In [ ]:
model.compile(
    optimizer=RMSprop(),
    loss="mse"
)

==========================================
Train Model
==========================================

In [ ]:
start_time = time.time()

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=125,
    verbose=0
)

In [ ]:
training_time = (
    time.time() - start_time
)

==========================================
Predictions
==========================================

In [ ]:
y_pred = model.predict(
    X_test_scaled
).flatten()

==========================================
Evaluation
==========================================

In [ ]:
mae, rmse, adjr2 = evaluate_model(
    y_test,
    y_pred,
    X.shape[1]
)

==========================================
Results Table
==========================================

In [ ]:
results_df = pd.DataFrame({

    "Optimizer": ["RMSProp"],

    "MAE": [mae],

    "RMSE": [rmse],

    "Adjusted_R2": [adjr2],

    "Training_Time_Seconds": [
        training_time
    ],

    "Final_Training_Loss": [
        history.history["loss"][-1]
    ],

    "Final_Validation_Loss": [
        history.history["val_loss"][-1]
    ]

In [ ]:
})

In [ ]:
results_df.to_csv(
    output_dir /
    "rmsprop_results.csv",
    index=False
)

==========================================
History
==========================================

In [ ]:
history_df = pd.DataFrame({

    "loss":
        history.history["loss"],

    "val_loss":
        history.history["val_loss"]

In [ ]:
})

In [ ]:
history_df.to_csv(
    output_dir /
    "rmsprop_history.csv",
    index=False
)

==========================================
Learning Curve
==========================================

In [ ]:
plt.figure(figsize=(10,6))

In [ ]:
plt.plot(
    history.history["loss"],
    label="Training Loss"
)

In [ ]:
plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

In [ ]:
plt.title(
    "RMSProp Learning Curve"
)

In [ ]:
plt.xlabel("Epoch")
plt.ylabel("Loss")

In [ ]:
plt.legend()
plt.grid(True)

In [ ]:
plt.savefig(
    output_dir /
    "rmsprop_learning_curve.png",
    bbox_inches="tight"
)

In [ ]:
plt.show()
plt.close()

==========================================
Actual vs Predicted
==========================================

In [ ]:
plt.figure(figsize=(8,8))

In [ ]:
plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

In [ ]:
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

In [ ]:
plt.title(
    "RMSProp: Actual vs Predicted"
)

In [ ]:
plt.xlabel(
    "Actual Quality"
)

In [ ]:
plt.ylabel(
    "Predicted Quality"
)

In [ ]:
plt.savefig(
    output_dir /
    "rmsprop_actual_vs_predicted.png",
    bbox_inches="tight"
)

In [ ]:
plt.show()
plt.close()

==========================================
Export Summary
==========================================

In [ ]:
with open(
    output_dir /
    "rmsprop_summary.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "Analysis 6 - RMSProp\n\n"
    )

    f.write(
        f"MAE: {mae:.4f}\n"
    )

    f.write(
        f"RMSE: {rmse:.4f}\n"
    )

    f.write(
        f"Adjusted R²: {adjr2:.4f}\n"
    )

    f.write(
        f"Training Time: {training_time:.2f} Seconds\n"
    )

==========================================
Final Output
==========================================

In [ ]:
print(
    "\n----- RMSPROP RESULTS -----"
)

In [ ]:
print(
    f"MAE: {mae:.4f}"
)

In [ ]:
print(
    f"RMSE: {rmse:.4f}"
)

In [ ]:
print(
    f"Adjusted R²: {adjr2:.4f}"
)

In [ ]:
print(
    f"Training Time: {training_time:.2f} Seconds"
)

In [ ]:
print(
    "\nAnalysis 6 - RMSProp Complete"
)